# 10.02 Daytona Sandbox

**`langchain-daytona`** 의 `DaytonaSandbox` 는 Daytona가 클라우드에서 관리하는 **Dev Container 기반 워크스페이스**를 Deep Agents `SandboxBackendProtocol` 로 감싼다.
Modal 과 달리 워크스페이스가 **파일시스템을 보존**하고, 스냅샷·이미지·볼륨을 통해 **재현성 있는 개발 환경** 을 제공한다.

## 학습 목표

- `Daytona()` 클라이언트 → `daytona.create(CreateSandboxFromImageParams(...))` → `DaytonaSandbox(sandbox=...)` 조립 패턴
- 이미지 지정 (`image="python:3.12-slim"`) vs 스냅샷(`CreateSandboxFromSnapshotParams`) 차이
- `.execute(cmd, timeout=...)` 의 **동기 폴링 인터벌** (`sync_polling_interval`) 의미
- 파일 I/O API (`read`, `write`, `upload_files`, `download_files`) — `ModalSandbox` 와 동일 시그니처
- `auto_stop_interval` · `auto_archive_interval` · `auto_delete_interval` 3단계 라이프사이클

## 언제 쓰나 — Modal vs Daytona vs Runloop

| 선택 | 적합 | 특징 |
|------|------|------|
| Modal | 스테이트리스한 burst, GPU | 서버리스, per-run 과금 |
| **Daytona** | **Dev Container** 이미지로 재현성 보장, 장기 워크스페이스, IDE 에이전트 | `.devcontainer.json` 호환, 스냅샷·볼륨·라벨 |
| Runloop | 세션 유지, 브라우저·셸 내장 | 코드 리뷰 루프, 상태 공유 |

## 10.02.1 환경 설정

필요 패키지: `langchain-daytona` (내부적으로 `daytona` SDK + deep-agents 를 끌고 온다).
`.env` 에는 Daytona Cloud 에서 발급한 `DAYTONA_API_KEY` 가 필요하다.

```bash
uv pip install langchain-daytona
# https://app.daytona.io 에서 API Key 발급 → .env 에 저장
```

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
assert os.environ.get("DAYTONA_API_KEY"), "DAYTONA_API_KEY 필요"

## 10.02.2 기본 사용 — 워크스페이스 한 개 띄우기

가장 짧은 E2E 경로.

1. `Daytona()` 클라이언트 생성 (`DAYTONA_API_KEY` 환경변수 자동 사용).
2. `daytona.create(CreateSandboxFromImageParams(image="python:3.12-slim"))` 로 워크스페이스 1개.
3. `DaytonaSandbox(sandbox=raw_sbx)` 래퍼로 감싸 `.execute("python -c 'print(1+1)'")`.
4. 명시 종료는 `raw_sbx.delete()` 또는 `auto_stop_interval` 로 위임.

In [ ]:
from daytona import Daytona, CreateSandboxFromImageParams
from langchain_daytona import DaytonaSandbox

client = Daytona()

raw_sbx = client.create(CreateSandboxFromImageParams(
    image="python:3.12-slim",
    language="python",
))

sandbox = DaytonaSandbox(sandbox=raw_sbx)
print("sandbox id:", sandbox.id)

r = sandbox.execute("python -c 'print(1+1)'")
print("exit:", r.exit_code, "stdout:", r.output.strip())

raw_sbx.delete()

## 10.02.3 `create_deep_agent(..., backend=DaytonaSandbox(...))` 연동

Modal 케이스와 똑같이, Deep Agents 는 backend 를 **단일 객체**로 받아 `ls` · `read_file` · `write_file` · 셸 실행을 전부 Daytona 로 라우팅한다.

Daytona 워크스페이스는 **파일시스템이 세션 간에 보존**되므로, 멀티턴 코드 편집·빌드 루프에 특히 유리하다.

In [ ]:
from deepagents import create_deep_agent

client = Daytona()

raw_sbx = client.create(CreateSandboxFromImageParams(
    image="python:3.12-slim",
    language="python",
    env_vars={"PYTHONUNBUFFERED": "1"},
    auto_stop_interval=15,     # 15분 유휴 시 자동 stop
))

backend = DaytonaSandbox(sandbox=raw_sbx)

agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    tools=[],
    system_prompt=(
        "너는 Daytona 워크스페이스 안에서 Python 스크립트를 작성·실행하는 엔지니어다. "
        "결과물은 /home/daytona/ 아래에 두고, 파일 목록과 주요 출력만 짧게 요약해서 답한다."
    ),
    backend=backend,
)

res = agent.invoke({"messages": [{"role": "user", "content": "fib.py 에 피보나치 10개를 출력하는 코드를 쓰고 실행해"}]})
print(res["messages"][-1].content)

raw_sbx.delete()

## 10.02.4 파일 전송 · 라이프사이클

`DaytonaSandbox` 의 파일 API 는 `ModalSandbox` · `RunloopSandbox` 와 **동일 시그니처**다 (5종).

- `write(path, content: str)` · `read(path)`
- `upload_files([(path, bytes), ...])` / `download_files([path, ...])`
- `edit(path, ...)` — 라인 단위 편집 (Deep Agents `edit_file` 도구 백엔드)

3단계 라이프사이클 (Daytona 특유):

| 필드 | 단위 | 의미 |
|------|------|------|
| `auto_stop_interval` | 분 | 유휴 시 **stop** — 컴퓨트 과금 중지, 파일시스템 유지 |
| `auto_archive_interval` | 분 | stop 이후 **archive** — 콜드 스토리지로 이동 (저렴한 보관) |
| `auto_delete_interval` | 분 | archive 이후 **삭제** |
| `ephemeral=True` | — | 세션 종료 시 즉시 완전 삭제 |

`DaytonaSandbox(sandbox=raw_sbx, timeout=1800, sync_polling_interval=0.1)` — 래퍼 자체의 기본 exec 타임아웃과 폴링 간격.

In [ ]:
client = Daytona()

raw_sbx = client.create(CreateSandboxFromImageParams(
    image="python:3.12-slim",
    language="python",
    auto_stop_interval=10,
    auto_archive_interval=60,
    auto_delete_interval=1440,
))

sandbox = DaytonaSandbox(sandbox=raw_sbx, timeout=600, sync_polling_interval=0.1)

# 텍스트 한 개 쓰기
sandbox.write("/home/daytona/note.txt", "hello from host")

# 배치 업로드 (bytes)
sandbox.upload_files([
    ("/home/daytona/data.csv", b"a,b\n1,2\n3,4\n"),
    ("/home/daytona/script.py", b"import csv\nrows = list(csv.reader(open('/home/daytona/data.csv')))\nprint(rows)\n"),
])

r = sandbox.execute("python /home/daytona/script.py")
print("exec exit:", r.exit_code, "output:", r.output.strip())

downloaded = sandbox.download_files(["/home/daytona/note.txt"])
print("downloaded:", downloaded[0].content)

raw_sbx.delete()

## 10.02.5 Dev Container 이미지 선택

Daytona 는 크게 **두 경로**로 환경을 지정한다.

### (a) 이미지 기반 — `CreateSandboxFromImageParams`

- `image="python:3.12-slim"` 같은 **Docker 이미지 태그** 문자열을 직접 지정
- 또는 `image=Image.debian_slim().pip_install(...).run_commands(...)` 로 **빌드 체인**을 코드로 선언 (Modal 스타일)
- `resources=Resources(cpu=2, memory=4, disk=10)` 로 리소스 한도 지정 가능

### (b) 스냅샷 기반 — `CreateSandboxFromSnapshotParams`

- 이미 빌드해 둔 **스냅샷 이름**(`snapshot="my-snap"`) 으로 즉시 복원 → 콜드스타트 단축
- 팀 공용 baseline(예: "팀의 python + poetry + 내부 라이브러리")을 스냅샷으로 저장해두면 cold start 수 초 → 수백 ms

### 공통 옵션

- `env_vars={"KEY": "VAL"}` — 컨테이너 환경변수 주입
- `labels={"team": "ml"}` — 검색용 라벨
- `network_block_all=True` 또는 `network_allow_list="github.com,pypi.org"` — 네트워크 정책
- `volumes=[VolumeMount(volume_id=..., mount_path="/data")]` — 영속 볼륨 마운트

In [ ]:
from daytona import CreateSandboxFromImageParams, Image as DImage

# 빌드 체인으로 이미지 선언 — pip 패키지를 미리 깔아둔 워크스페이스
custom_image = (
    DImage.debian_slim("3.12")
    .pip_install(["requests", "pandas"])
    .run_commands("echo 'ready' > /etc/motd")
)

client = Daytona()
raw_sbx = client.create(CreateSandboxFromImageParams(
    image=custom_image,
    language="python",
    env_vars={"APP_ENV": "sandbox"},
    labels={"purpose": "lc-notebook"},
    network_allow_list="pypi.org,files.pythonhosted.org",
    auto_stop_interval=10,
))

sandbox = DaytonaSandbox(sandbox=raw_sbx)
print(sandbox.execute("python -c 'import pandas; print(pandas.__version__)'").output.strip())
print(sandbox.execute("cat /etc/motd").output.strip())

raw_sbx.delete()

## 10.02.6 비용 · 지연 트레이드오프

| 축 | Daytona | Modal | Runloop |
|----|---------|-------|---------|
| 콜드스타트 | 수 초 (스냅샷 경로 시 수백 ms) | 수 초 | 수 초 |
| per-exec 지연 | ~100ms (내부 exec API) | ~100ms | ~100ms |
| 격리도 | **워크스페이스 컨테이너** | 컨테이너 | 컨테이너 |
| 파일시스템 | **stop 후에도 보존** → archive → delete | 샌드박스 종료 시 휘발 | 세션 유지 |
| 비용 | 실행 + 파일시스템 보관 | 실행만 | 세션 과금 |
| 적합 | 재현성 있는 **개발 워크스페이스** | 스테이트리스 burst, GPU | 장시간 상호작용 세션 |

설계 힌트:

- 팀이 반복 사용하는 환경은 **스냅샷**으로 저장 → cold start 단축 + 일관성 확보.
- 민감 파일·중간 산출물을 보관해야 하면 Modal 대신 Daytona 가 적합 (Modal 샌드박스는 종료 시 휘발).
- `auto_stop_interval` 을 공격적으로 잡으면 유휴 과금을 크게 줄일 수 있다. 사용자가 다시 보내는 요청은 Daytona 가 자동으로 **wake** 시킨다.

## 체크리스트

- [ ] `.env` 에 `DAYTONA_API_KEY` 존재
- [ ] `Daytona().create(CreateSandboxFromImageParams(image=...))` — 이미지 문자열 또는 `Image.debian_slim()` 체인
- [ ] `DaytonaSandbox(sandbox=raw_sbx, timeout=..., sync_polling_interval=0.1)` — 래퍼 옵션
- [ ] `auto_stop_interval` / `auto_archive_interval` / `auto_delete_interval` — 3단 라이프사이클 설정
- [ ] 신뢰 불가 코드에는 `network_block_all=True` 또는 allow-list
- [ ] `create_deep_agent(..., backend=DaytonaSandbox(...))` — 에이전트의 파일·셸이 Daytona 로 라우팅

## 다음

- `04_deepagents/10_sandboxes_and_acp.ipynb` — Deep Agents × 샌드박스 실전
- `07_integration/10_sandboxes/01_modal.ipynb` — 서버리스 burst + GPU
- `07_integration/10_sandboxes/03_runloop.ipynb` — 세션형 샌드박스

## 참고

- `langchain-daytona` PyPI: https://pypi.org/project/langchain-daytona/
- Daytona 문서: https://www.daytona.io/docs
- Daytona Dev Container 가이드: https://containers.dev/